# ✍️ OCR الخط اليدوي (عربي + إنجليزي + ألماني) — من HF Bucket إلى نموذج مدرَّب (Colab T4)

هذه النوت بوك تمر على خطوات الحزمة بالترتيب:

1. **تجهيز** الحزمة والمكتبات
2. **تسجيل دخول** Hugging Face آمنًا (Secrets — بدون لصق رموز في الكود)
3. **تنزيل** محتويات الـ bucket `DrAbdulmalek/github-repo-backups/2026-10-14`
4. **فحص** النسخ الاحتياطية وتجهيز صفحات الخط اليدوي
5. **قص ووسم** الكلمات (أداة Gradio) → قاعدة بيانات SQLite
6. **تصدير** مجموعة البيانات وتقسيمها على مستوى الصفحة
   - بينهما: **6ب** مراجعة الوسوم و **6ج** تعديل مربعات الكلمات على الصفحات
7. **تدريب** نموذج TrOCR (ViT + فك bert-base-multilingual-cased: ar/en/de) واختباره

> 🔐 **أمن الرموز:** استخدم رمزًا **جديدًا** من huggingface.co/settings/tokens.
> رمز سبق لصقه في ملف أو دردشة يُعتبر مكشوفًا — ألغه فورًا. لا تضع الرمز في أي خلية إطلاقًا.

In [ ]:
# 1) رفع الحزمة إلى Colab (ارفع ملف arabic-handwriting-ocr-toolkit.zip)
import pathlib, zipfile

PKG = "arabic-handwriting-ocr-toolkit"
if not pathlib.Path(PKG).exists():
    from google.colab import files
    up = files.upload()                     # اختر arabic-handwriting-ocr-toolkit.zip
    zname = [k for k in up if k.endswith(".zip")][0]
    with zipfile.ZipFile(zname) as zf:
        zf.extractall(".")
print(f"✓ الحزمة جاهزة في {pathlib.Path(PKG).resolve()}")

In [ ]:
# 2) المكتبات (حدّث huggingface_hub أولًا — ميزة Buckets حديثة)
%cd arabic-handwriting-ocr-toolkit
!pip install -q -U huggingface_hub
!pip install -q -r requirements.txt

In [ ]:
# 3) تسجيل دخول Hugging Face — من Colab Secrets أو إدخال مخفي
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✓ تم قراءة HF_TOKEN من Colab Secrets")
except Exception:
    from getpass import getpass
    os.environ["HF_TOKEN"] = getpass("الصق رمز Hugging Face (لن يظهر في المخرجات): ")
print("✓ الرمز جاهز في متغير البيئة — لا يُكتب في أي ملف")

## 4) تنزيل الـ bucket ثم فحص النسخ وتجهيز الصفحات

In [ ]:
# 4) تنزيل محتويات الـ bucket (المجلد 2026-10-14)
!python 1_download_bucket.py

In [ ]:
# 5) فحص النسخ الاحتياطية: استخراج آمن + PDF → صور + تجهيز ocr_data/pages
!python 2_inspect_backups.py

## 6) قص ووسم الكلمات

شغّل الخلية التالية، افتح رابط Gradio، ثم لكل صفحة: انقر **زاويتي الكلمة** ←
اكتب نصها ← «حفظ القصاصة» ← «إنهاء الصفحة».

> 🛑 عند الانتهاء **أوقف هذه الخلية بزر Stop** ثم تابع الخلايا التالية.

In [ ]:
# 6) أداة القص والوسم (Gradio + SQLite)
!python 3_labeling_app.py

In [ ]:
# 6ب) مراجعة الوسوم الآلية المستوردة من قاعدة بيانات مشروعك (حالة auto)
# تعرض الأقل ثقة أولًا: ✅ قبول | 💾 تصحيح | 🚫 تجاهل | ⏭️ تخطي | ↩️ تراجع
# القرارات تُحفظ فورًا في crops.db — والتصدير (الخطوة التالية) يمرّ كل وسم
# مُراجَع بشريًا (verified=1) ولو كان قصيرًا (ja, im, zu…) ويستبعد المتجاهلة.
!python 3b_review_app.py


## 6ج) مراجعة وتعديل مربعات الكلمات على الصفحات

يعرض كل صفحة **كاملة** أو **سطرًا** أو **جزءًا مكبّرًا** مع مربعات الكلمات، ويشغّل خادمًا محليًا:
- الخلية تنزّل البيانات تلقائيًا من المستودع الخاص إن لم توجد، وتطبع رابطي **المحرر** و**اللوحة**.
- اسحب **حواف/زوايا المربع** لتوسيعه أو تضييقه (التقسيم الآلي قد يقطع الكلمة أو يضمّ أكثر من كلمة).
- **✂ تقسيم** مربع يضم كلمتين، **دمج مع الجار** لكلمة مقسومة على مربعين، **➕ مربع جديد** لكلمة لم تُلتقط.
- الحفظ تلقائي في `crops.db` + سجل تدقيق `box_adjustments.jsonl`، وصورة القصاصة تُعاد قصّها آليًا.
- بعد الانتهاء: **أوقف الخلية** ثم أعد تشغيل خلية **7) التصدير** لتحديث مجموعة التدريب.

In [ ]:
# 6ج) خادم تعديل المربعات — يجهّز البيانات تلقائيًا ثم يطبع رابط الواجهة
import os, subprocess, sys, time, urllib.request, tarfile, shutil, urllib.parse
from pathlib import Path

PORT = 7863
if not Path("ocr_data/crops.db").exists():
    print("⏳ تنزيل البيانات من المستودع الخاص (القاعدة + اللوحات + الأسطر + صور القصاصات)…")
    from huggingface_hub import snapshot_download
    snapshot_download("DrAbdulmalek/handwriting-ocr-crops", repo_type="dataset",
                      local_dir="ocr_data",
                      allow_patterns=["crops.db", "notes.db", "repo_pages/*", "crops.tar.gz"])
    d = Path("ocr_data/extracted/handwriting-notes-meta/handwriting-notes/data")
    d.mkdir(parents=True, exist_ok=True)
    if Path("ocr_data/notes.db").exists() and not (d / "notes.db").exists():
        shutil.move("ocr_data/notes.db", d / "notes.db")
    tar = Path("ocr_data/crops.tar.gz")
    if tar.exists():
        with tarfile.open(tar, "r:gz") as zf:
            zf.extractall("ocr_data")
        tar.unlink()
    print("✓ البيانات جاهزة")

# احصل على origin الوكيل قبل تشغيل الخادم، واسمح بهذا الأصل المحدد فقط.
proxy_base = None
server_env = os.environ.copy()
try:
    from google.colab import output
    proxy_base = output.eval_js(f"google.colab.kernel.proxyPort({PORT})").rstrip("/") + "/"
    parsed_proxy = urllib.parse.urlsplit(proxy_base)
    if parsed_proxy.scheme not in ("http", "https") or not parsed_proxy.netloc:
        raise ValueError("Colab proxy returned an invalid URL")
    server_env["BOX_EDITOR_ALLOWED_ORIGINS"] = f"{parsed_proxy.scheme}://{parsed_proxy.netloc}"
except Exception:
    proxy_base = None

proc = subprocess.Popen(
    [sys.executable, "7_box_editor_server.py", "--port", str(PORT)],
    env=server_env,
)
for _ in range(60):
    try:
        # / is a static shell; API health now correctly requires the access token.
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1).read(1)
        break
    except Exception:
        time.sleep(1)
print(f"✓ الخادم يعمل على المنفذ {PORT}")
if proxy_base:
    print("✏️ واجهة تعديل المربعات:", proxy_base)
    print("🎛️ لوحة التحكم:", proxy_base + "panel")
else:
    print(f"🌐 افتح الواجهة: http://127.0.0.1:{PORT}/panel")
print("🔐 رمز الوصول محفوظ في ~/.config/arabic-handwriting-ocr-toolkit/box-editor.token")
print("لتنزيل ملف الرمز إلى جهازك من Colab عند الحاجة:")
print("from google.colab import files; files.download(str(Path.home() / '.config/arabic-handwriting-ocr-toolkit/box-editor.token'))")
# 🛑 للإيقاف لاحقًا:  proc.terminate()


## 7) التصدير والتدريب والتجربة

In [ ]:
# 7) SQLite → مجموعة بيانات HF (تقسيم على مستوى الصفحة، بلا تسريب)
!python 4_export_dataset.py

In [ ]:
# 8) تدريب TrOCR (ViT + فك bert-base-multilingual-cased: ar/en/de) على T4
# عند OOM: أضف  --batch-size 4 --image-size 224
!python 5_train_trocr_colab.py --epochs 20

In [ ]:
# 9) تجربة سريعة على قصاصاتك
!python 6_predict.py --model trocr-multilingual-handwritten --image ocr_data/crops --max 20

In [ ]:
# 10) (اختياري) رفع النموذج إلى Hugging Face Hub
# استخدم رمز Write مؤقتًا في Colab Secrets ثم ألغه فورًا بعد الرفع
import os
REPO = ""   # مثال: "DrAbdulmalek/trocr-multilingual-handwritten"

if REPO:
    !python 5_train_trocr_colab.py --epochs 0 --push-repo {REPO}
else:
    print("تم التخطي — اكتب اسم المستودع في REPO للرفع")

## ملاحظات ترفع الجودة

- **كمية:** استهدف 500–2000 قصاصة موسومة على الأقل؛ التنوع بين الصفحات أهم من عددها.
- **التقسيم:** تم على مستوى الصفحة تلقائيًا — لا تربت يدويًا في CSV إلا وهوامش مدروسة.
- **الرموز:** بعد أي رفع إلى Hub، ألغِ رمز الكتابة من إعدادات الحساب.
- **التفاصيل الكاملة** (استكشاف الأخطاء، البنى، الترخيص): ملف `README.md` في الحزمة.